## Install Dependencies

`cartopy` and geospatial imaging libraries occasionally require system-level C libraries. We install those first, followed by the Python packages.

In [ ]:
!apt-get install -y libgeos-dev libproj-dev

In [ ]:
!pip install -q arraylake xarray microsoft-aurora huggingface_hub matplotlib pcodec cartopy imageio

## Secrets

Stored as `HF_TOKEN` and `ARRAYLAKE_TOKEN`

In [ ]:
import os
from google.colab import userdata
from huggingface_hub import login as hf_login
from arraylake import Client

# huggingface login
try:
  hf_token = userdata.get("HF_TOKEN")
except Exception:
  hf_token = input("Enter your Hugging Face token: ")

hf_login(token=hf_token)

# arraylake login
try:
  arraylake_token = userdata.get("ARRAYLAKE_TOKEN")
except Exception:
  arraylake_token = input("Enter your ArrayLake token: ")

client = Client(token=arraylake_token)

print("Connecting to Hugging face and Arraylake............")
print("")
print("Done!")


## Fetch ERA5 initialization data from Earthmover

Aurora requires initial conditions for both the target prediction start time and the prior 6-hour interval.

In [ ]:
import pickle
import xarray as xr
from huggingface_hub import hf_hub_download

# Open the public ERA5 Zarr store via Arraylake session
era5_repo = client.get_repo("earthmover-public/era5")
rsession = era5_repo.readonly_session("main")

# Target timestamp + previous 6h interval
# Note: Earthmover's public archive contains historical ERA5 data. Select a date within the archive window:
TIMESTAMPS = ["2024-03-12T18:00:00", "2024-03-13T00:00:00"]

In [ ]:
print("Streaming surface-level variables...")
era5_single = xr.open_zarr(rsession.store, group="single/spatial", chunks=None)
surf_data = era5_single.sel(valid_time=TIMESTAMPS)[
    ["u10", "v10", "d2m", "t2m", "msl", "skt", "sp", "tcw", "stl1", "stl2", "swvl1", "swvl2"]
]

In [ ]:
print("Streaming pressure-level variables...")
era5_pressure = xr.open_zarr(rsession.store, group="pressure/spatial", chunks=None)
atmos_data = era5_pressure.sel(valid_time=TIMESTAMPS)

In [ ]:
print("Downloading static variables from Hugging Face Hub...")
static_path = hf_hub_download(repo_id="microsoft/aurora", filename="aurora-0.25-static.pickle")
with open(static_path, "rb") as f:
    static_vars = pickle.load(f)

Data retrieval completed at this point!!!!

## Assemble Aurora Input Batch

Format the ERA5 multidimensional arrays into the PyTorch tensor structures expected by Aurora's forward rollout.

In [ ]:
import datetime
import numpy as np
import torch
from aurora import Batch, Metadata

In [ ]:
def to_tensor(arr):
  # add batch dimension and convert to float32 contiguous tensor
  return torch.from_numpy(np.ascontiguousarray(arr, dtype=np.float32))[None]

In [ ]:
# surface variables mapping (Aurora format: ERA5 variable name)
surf_map = {"2t": "t2m", "10u": "u10", "10v": "v10", "msl": "msl"}
atmos_vars = ["t", "u", "v", "q", "z"]

In [ ]:
batch = Batch(
    surf_vars={k: to_tensor(surf_data[v].values) for k, v in surf_map.items()},
    static_vars={k: to_tensor(v)[0] for k, v in static_vars.items()},
    atmos_vars={k: to_tensor(atmos_data[k].values) for k in atmos_vars},
    metadata=Metadata(
        lat=torch.from_numpy(era5_single.latitude.values.astype("f4")),
        lon=torch.from_numpy(era5_single.longitude.values.astype("f4")),
        time=(datetime.datetime.fromisoformat(TIMESTAMPS[-1]),),
        atmos_levels=tuple(era5_pressure.pressure_level.values.tolist()),
    ),
)

In [ ]:
print("Batch created:")
print(f"- Surface vars: {list(batch.surf_vars.keys())}")
print(f"- Atmos vars: {list(batch.atmos_vars.keys())}")

## Load Weight and Run autoregressive rollout

In [ ]:
from aurora import Aurora, rollout

# load the pretrained 0.25-degree model checkpoint
print("Loading aurora checkpoint...")
model = Aurora(use_lora=False)
model.load_checkpoint("microsoft/aurora", "aurora-0.25-pretrained.ckpt")
model.eval()

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
print(f"Model loaded and dispatched to: {device}")

In [ ]:
STEPS = 4
print(f"Running {STEPS}-step autoregressive rollout...")

with torch.inference_mode():
    # Move model and batch to CPU to avoid OutOfMemoryError on GPU
    model_cpu = model.to("cpu")
    batch_cpu = batch.to("cpu")
    predictions = [pred.to("cpu") for pred in rollout(model_cpu, batch_cpu, steps=STEPS)]

print("Inference completed successfully!")

## Convert predictions back to Xarray dataset

In [ ]:
import pandas as pd

dims = ["init_time", "step", "latitude", "longitude"]
meta = predictions[0].metadata

In [ ]:
# Merge each forecast step into an xarray dataset
predictions_ds = xr.concat(
    [
        xr.Dataset(
            data_vars={
                v: (dims, predictions[i].surf_vars[v].numpy())
                for v in predictions[0].surf_vars
            }
        )
        for i in range(STEPS)
    ],
    dim="step"
)

In [ ]:
# Attach spatial and temporal coordinates
predictions_ds = predictions_ds.assign_coords(
    longitude=meta.lon,
    latitude=meta.lat,
    init_time=[meta.time[0]],
    step=[pd.Timedelta(hours=6) * i for i in range(STEPS)]
)

predictions_ds = predictions_ds.assign_coords(
    valid_time=predictions_ds.init_time + predictions_ds.step
)
predictions_ds = predictions_ds.squeeze().swap_dims({"step": "valid_time"})

print("Resulting Forecast Dataset:")
print(predictions_ds)

## Render and Display an Animated Globe

This cell extracts the 2-meter surface temperature (`2t`), converts Kelvin to Celsius, plots an orthographic globe, and creates a rotating GIF directly inside your notebook.

In [ ]:
import io
import imageio.v3 as iio
import matplotlib as mpl
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
from IPython.display import Image as IPImage, display

In [ ]:
# Convert 2m temperature from Kelvin to Celsius
temp_c = predictions_ds["2t"] - 273.15
vmin, vmax = float(np.nanpercentile(temp_c.values, 2)), float(np.nanpercentile(temp_c.values, 98))
norm = mpl.colors.Normalize(vmin=vmin, vmax=vmax)

frames = []
frames_per_step = 10
total_steps = len(temp_c.valid_time)

In [ ]:
print("Generating globe frames...")
for step_idx in range(total_steps):
    step_data = temp_c.isel(valid_time=step_idx)
    valid_time_str = pd.Timestamp(step_data.valid_time.values).strftime("%Y-%m-%d %H:%M UTC")

    for sub_frame in range(frames_per_step):
        # Progressively rotate central longitude for animation
        lon_rot = (step_idx * frames_per_step + sub_frame) * (360.0 / (total_steps * frames_per_step))

        fig = plt.figure(figsize=(7, 6), dpi=100)
        ax = fig.add_axes([0.05, 0.08, 0.72, 0.84], projection=ccrs.Orthographic(central_longitude=lon_rot, central_latitude=15))
        ax.set_global()
        ax.coastlines(color="black", linewidth=0.8)

        step_data.plot(
            ax=ax,
            transform=ccrs.PlateCarree(),
            vmin=vmin,
            vmax=vmax,
            cmap="RdYlBu_r",
            add_colorbar=False,
            add_labels=False
        )

        ax.set_title(f"Aurora Forecast: 2m Temp (°C)\nValid: {valid_time_str}", fontsize=11)

        cax = fig.add_axes([0.80, 0.15, 0.03, 0.7])
        fig.colorbar(mpl.cm.ScalarMappable(norm=norm, cmap="RdYlBu_r"), cax=cax, label="Temperature [°C]")

        buf = io.BytesIO()
        plt.savefig(buf, format="png", bbox_inches="tight")
        plt.close(fig)
        buf.seek(0)
        frames.append(iio.imread(buf))

gif_path = "aurora_forecast.gif"
iio.imwrite(gif_path, frames, duration=100, loop=0)
print("Rendering complete!")

display(IPImage(filename=gif_path))